# How Labs 01–05 Fit Together
AI Cybersecurity Masterclass · DSJ THE CRITTERS

This isn't a sixth lab — it's the map. Each notebook in this repo answers a specific question, and the order isn't arbitrary: each lab exists *because* of a limitation the previous one exposed. Read this before (or after) working through 01–05 to see why the sequence is the sequence, not just what each notebook does.

In [ ]:
from pathlib import Path
import sys, os

try:
    import google.colab
    if not Path("ai-cybersecurity-public-labs").exists():
        !git clone https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git
    os.chdir("ai-cybersecurity-public-labs")
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)

## Grounding: what the data actually looks like

Every lab below works with the same UNSW-NB15 files. One quick recap so the reasoning that follows isn't abstract.

In [ ]:
from src.data_loader import load_unsw

train, test = load_unsw(DATA_DIR)
label_counts = train["label"].value_counts()
rarest = train.loc[train["label"] == 1, "attack_cat"].value_counts().sort_values().head(2)

print(f"{train.shape[0]:,} training rows, {train.shape[1]} columns")
print(f"Label split: {label_counts.get(1,0):,} attack / {label_counts.get(0,0):,} normal")
print(f"Rarest attack categories: {dict(rarest)}")

---

## Lab 01 — Security Data Exploration
**The question it answers:** *what am I actually working with?*

This comes first for one reason: every later notebook trusts `label` and `attack_cat` without re-checking them. If you skip this step, you inherit whatever's wrong with the data silently — a leaked `id` column, an attack-heavy class balance nobody flagged, categories so rare a model can't learn them. Lab 01 is where those get caught, before they become someone else's bug three notebooks later.

The 68/32 attack-heavy split and the fact that Worms has only 130 rows aren't footnotes — they're the reason Labs 02 and 03 are built the way they are.

## Lab 02 — Binary Threat Detection
**The question it answers:** *can a baseline model separate attack from normal at all?*

This is deliberately the *easiest possible version* of the modeling problem — two classes, one of which has 119,341 examples. Binary comes before multiclass for the same reason you test a smoke detector before wiring a full alarm system: if a baseline can't even manage the easy version of this task, there's no point adding nine attack categories on top of it.

Lab 02's real job isn't the accuracy number — it's establishing what "a working baseline" looks like, so Lab 03 has something to be harder *than*.

## Lab 03 — Multiclass Attack Classification
**The question it answers:** *does the model still work once "attack" isn't one thing?*

This is where the rare categories from Lab 01 stop being an observation and start being a problem you can measure. Worms at 130 rows either gets recognized here or it doesn't — and the gap between macro and weighted averages is the notebook's real point: weighted F1 can look fine while the model is quietly failing on every category that isn't Generic or Exploits. Binary accuracy from Lab 02 can't show you that at all. This is why multiclass comes second, not first — you need the binary baseline to know a two-class problem was never going to reveal this failure mode.

## Lab 04 — Feature Engineering for Cybersecurity
**The question it answers:** *now that I've seen where the model fails, what would actually help?*

This comes *after* Labs 02 and 03, not before, on purpose. Feature engineering without knowing your failure modes is guessing — you'd be building features hoping they help, instead of building features aimed at the specific gaps Lab 03 just measured (weak recall on rare categories, particular confusions between attack types). Doing this step first would mean redoing it once the real weaknesses showed up anyway.

## Lab 05 — Model Comparison, Thresholds & Errors
**The question it answers:** *given everything above, which model actually deploys — and how do I know?*

This is last because comparison only means something once "good" has a definition. Without Labs 01–04, "Model A beats Model B" is just a leaderboard. With them, you already know the base rate is attack-heavy (01), what a baseline can and can't do (02), which categories are structurally hard (03), and what features might move the needle (04) — so Lab 05's comparison is being read against real context, not in a vacuum. This is also where accuracy stops being the deciding metric and *which model fails least dangerously* takes over — the same question the accuracy-trap lesson opens with.

---

## The throughline, in one line each

| Lab | Question | Depends on |
|---|---|---|
| 01 | What am I working with? | Nothing — this is the foundation |
| 02 | Can a baseline separate attack from normal? | 01's class balance |
| 03 | Does it still work with real attack categories? | 02's baseline, 01's rare classes |
| 04 | What features would actually help? | 03's specific failure modes |
| 05 | Which model do I deploy, and why? | Everything above |

Skipping a step doesn't just lose that notebook's content — it removes the context the next one is reasoning against.